# Word2Vec: Skip-gram и CBOW на PyTorch

Обучаются две учебные реализации Word2Vec на PyTorch:
**Skip-gram** и **CBOW**.

## Библиотеки и параметры

In [ ]:
import re
import math
import random
import collections
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "word2vec"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


 
print("Папка для графиков:", FIGURES_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

EMBEDDING_DIM = 100
WINDOW_SIZE = 4
NEG_SAMPLES = 8
MIN_FREQ = 2
BATCH_SIZE = 64
EPOCHS = 25
HS_EPOCHS = 6
LEARNING_RATE = 3e-3
NS_ALPHA = 0.75
SUBSAMPLE_T = 1e-4
USE_SUBSAMPLING = False

DEVICE = torch.device('cpu')
print(f'Устройство: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Используем subsampling: {USE_SUBSAMPLING}')

## Подготовка корпуса

In [ ]:
RAW_CORPUS = """
The sun rises in the east and sets in the west every single day.
Birds fly south in the winter to escape the cold weather.
Lions are the kings of the African savanna and hunt in packs.
The ocean covers more than seventy percent of the earth surface.
Trees absorb carbon dioxide and release oxygen into the atmosphere.
Dogs are loyal companions that have lived with humans for thousands of years.
Wolves hunt in coordinated packs and communicate through howls.
The mountain peaks are covered with snow during winter months.
Rivers flow from mountains to the sea carrying fresh water.
Elephants are the largest land animals and have excellent memory.
Neural networks are inspired by the structure of the human brain.
Machine learning algorithms learn patterns from large datasets.
Deep learning models have many layers that extract hierarchical features.
Natural language processing enables computers to understand human text.
Transformers revolutionized the field of natural language processing.
Convolutional networks excel at image recognition and computer vision tasks.
Recurrent networks process sequential data like text and time series.
Attention mechanisms allow models to focus on relevant parts of input.
Gradient descent optimizes model parameters by minimizing loss functions.
Backpropagation computes gradients efficiently through the network layers.
Word embeddings represent words as dense vectors in continuous space.
Semantic similarity between words can be measured using cosine distance.
Vocabulary size affects the computational cost of language models.
Tokenization splits raw text into smaller units called tokens.
Language models predict the next word given a sequence of previous words.
Cats are independent animals that sleep for many hours each day.
Fish swim through rivers and oceans using their fins and tails.
Eagles soar high above mountains and hunt small animals below.
Dolphins are intelligent marine mammals that communicate with each other.
Bears hibernate during winter and wake in spring to find food.
The forest provides habitat for thousands of animal and plant species.
Rainfall is essential for plants to grow and rivers to flow.
Deserts are arid regions where very little rain falls each year.
Coral reefs are underwater ecosystems rich with marine biodiversity.
Bees pollinate flowers and produce honey in their hive colonies.
The water cycle includes evaporation condensation and precipitation phases.
Volcanoes erupt and release lava ash and gases into the environment.
Earthquakes occur when tectonic plates shift beneath the earth surface.
The atmosphere protects earth from harmful radiation from the sun.
Seasons change as the earth orbits around the sun each year.
Artificial intelligence is transforming industries and changing how people work.
Computers process information using binary code of zeros and ones.
The internet connects billions of devices and people around the world.
Software algorithms solve complex problems through a series of steps.
Data science combines statistics and programming to extract insights.
Cloud computing allows storage and processing of data on remote servers.
Cybersecurity protects computer systems and networks from malicious attacks.
Robots are programmed machines that can perform physical tasks automatically.
Sensors collect data from the physical world for computer processing.
Databases store and organize large amounts of structured information.
Cheetahs are the fastest land animals running at incredible speeds.
Penguins live in cold Antarctic regions and swim to hunt fish.
Whales are the largest animals on earth and live in oceans.
Gorillas are great apes that live in the forests of Africa.
Crocodiles are ancient reptiles that live near rivers and swamps.
Butterflies transform from caterpillars through a process called metamorphosis.
Ants build complex underground colonies and work together as a team.
Spiders spin webs to catch insects for their daily food.
Owls hunt at night using their exceptional vision and hearing.
Pandas eat bamboo in the forests of China and are endangered.
The Amazon rainforest is the largest tropical forest on the planet.
Glaciers are massive bodies of ice that move slowly over land.
Wetlands filter water and provide habitat for many aquatic species.
Mangrove forests protect coastlines from erosion and storm damage.
Grasslands support large herds of grazing animals like zebras and bison.
The tundra is a cold biome with permafrost and few trees.
Biodiversity refers to the variety of life forms in an ecosystem.
Photosynthesis converts sunlight and carbon dioxide into glucose and oxygen.
Ecosystems maintain balance through complex food chains and energy flows.
Climate change threatens ecosystems and the survival of many species.
Renewable energy sources like solar and wind reduce carbon emissions.
Solar panels convert sunlight directly into electrical energy efficiently.
Wind turbines generate electricity by harnessing the power of wind.
Electric vehicles reduce dependence on fossil fuels and lower emissions.
Batteries store electrical energy for later use in devices and vehicles.
Nuclear energy releases tremendous power by splitting atomic nuclei.
Hydropower generates electricity from the movement of flowing water.
Geothermal energy uses heat from beneath the earth surface for power.
Biofuels are made from organic materials like plants and agricultural waste.
Energy efficiency reduces consumption and lowers environmental impact.
The human brain contains billions of neurons connected by synapses.
Memory is stored through the strengthening of synaptic connections.
Language is a uniquely human ability that allows complex communication.
Emotions influence decision making and social behavior in humans.
Sleep is essential for brain health memory consolidation and recovery.
Exercise improves physical and mental health by releasing endorphins.
Nutrition provides the energy and building blocks the body needs.
Genetics determines many physical and behavioral traits in living organisms.
Evolution occurs through natural selection over many generations of time.
Cells are the fundamental building blocks of all living organisms.
DNA carries genetic information that is passed from parents to children.
Proteins are large molecules that perform most functions in the cell.
The immune system defends the body against bacteria viruses and disease.
Antibiotics kill or inhibit the growth of harmful bacteria in body.
Vaccines train the immune system to fight specific infectious diseases.
The heart pumps blood through the circulatory system continuously.
Lungs exchange oxygen and carbon dioxide during the breathing process.
Muscles contract and relax to produce movement throughout the body.
The skeleton provides structure and protection for organs and tissues.
Foxes are clever predators that adapt to many different environments.
Deer graze in forests and meadows and are hunted by wolves.
Rabbits reproduce quickly and are an important prey animal in ecosystems.
Squirrels collect and store nuts and seeds for the winter season.
Hedgehogs roll into a ball when threatened by predators.
Bats are the only mammals capable of sustained powered flight.
Frogs live both on land and in water and eat insects.
Snakes move without legs and hunt using heat sensing organs.
Turtles carry their homes on their backs and live very long lives.
Crows are highly intelligent birds that use tools to find food.
Programming languages allow humans to give instructions to computers.
Python is a popular language used in data science and machine learning.
Version control systems like git track changes in code over time.
APIs allow different software systems to communicate and share data.
Microservices break applications into small independent deployable components.
Containers package software and its dependencies for consistent deployment.
DevOps practices improve collaboration between development and operations teams.
Testing ensures software behaves correctly and meets quality requirements.
Documentation helps developers understand and maintain complex codebases.
Open source software is freely available for anyone to use and modify.
Stars are massive balls of plasma that generate light and heat.
Galaxies contain billions of stars held together by gravitational force.
Black holes are regions where gravity is so strong light cannot escape.
The moon orbits the earth and influences ocean tides daily.
Planets orbit stars in elliptical paths due to gravitational attraction.
Comets are icy bodies that release gas and dust near the sun.
Asteroids are rocky bodies that orbit the sun in the asteroid belt.
Light travels at about three hundred thousand kilometers per second.
The universe is approximately fourteen billion years old and expanding.
Telescopes allow scientists to observe distant objects in the universe.
Mathematics provides the language and tools for scientific reasoning.
Statistics is the science of collecting analyzing and interpreting data.
Probability measures the likelihood of events occurring in random processes.
Linear algebra deals with vectors matrices and systems of linear equations.
Calculus studies change and motion through derivatives and integrals.
Graph theory models relationships between objects as nodes and edges.
Optimization finds the best solution among many possible alternatives.
Information theory studies the quantification storage and communication of data.
Cryptography protects information through mathematical encoding techniques.
Algorithms are step by step procedures for solving computational problems.
Physics studies the fundamental laws that govern matter and energy.
Chemistry investigates the composition structure and properties of matter.
Biology examines the structure function growth and evolution of life.
Geology studies the solid earth its rocks and the processes that shape it.
Astronomy is the scientific study of celestial objects and phenomena.
Ecology examines how organisms interact with each other and their environment.
Psychology studies human behavior thought and emotion scientifically.
Economics analyzes how societies allocate scarce resources efficiently.
History records and interprets past human events and civilizations.
Philosophy explores fundamental questions about existence knowledge and ethics.
Art expresses human creativity and emotion through visual forms.
Music combines sound rhythm and harmony to create emotional experiences.
Literature uses language as an art form to tell stories and ideas.
Architecture designs structures that balance function beauty and durability.
Medicine applies scientific knowledge to maintain and restore human health.
Engineering applies science and math to design and build useful things.
Agriculture produces food through cultivation of plants and raising animals.
Transportation moves people and goods from one location to another.
Communication technology connects people across vast distances instantly.
Manufacturing transforms raw materials into finished products efficiently.
Trade exchanges goods and services between people regions and countries.
Education transmits knowledge skills and values to new generations.
Research advances human knowledge by asking and answering new questions.
Innovation creates new solutions to problems by combining ideas creatively.
Cooperation allows individuals to achieve goals that are impossible alone.
Language enables humans to share abstract thoughts and complex ideas.
Culture encompasses the shared beliefs practices and values of a group.
Society is organized through institutions norms and social relationships.
Democracy allows citizens to participate in making collective decisions.
Law establishes rules that govern behavior and resolve conflicts fairly.
Ethics guides moral choices about what is right wrong and just.
Sustainability meets present needs without compromising future generations.
Diversity strengthens communities and organizations through varied perspectives.
Empathy allows people to understand and share the feelings of others.
Creativity generates novel ideas by combining existing knowledge in new ways.
Critical thinking evaluates claims and arguments based on evidence and logic.
Problem solving applies knowledge and reasoning to overcome challenges.
Collaboration combines the strengths of different people to achieve goals.
Leadership inspires and guides others toward shared goals and visions.
Communication conveys information feelings and ideas to other people.
Learning is the process of acquiring new knowledge skills and understanding.
Growth requires effort practice and the willingness to make mistakes.
Curiosity drives exploration and the pursuit of new knowledge.
Patience allows difficult tasks to be completed successfully over time.
Perseverance continues effort despite obstacles setbacks and failures.
Kindness creates positive relationships and builds trust between people.
Respect acknowledges the value and dignity of every person.
Honesty builds trust and is essential for healthy relationships.
Courage allows people to act rightly even in the face of fear.
Wisdom applies knowledge and experience to make good judgments.
Balance ensures that multiple important areas of life receive attention.
The wolf is a predator that roams forests and hunts deer and elk.
Tigers are solitary big cats that stalk prey in dense jungles.
Monkeys live in trees and are social animals with complex behaviors.
Parrots can mimic human speech and are highly intelligent birds.
Horses have been used by humans for transportation and farming for centuries.
Cows provide milk and meat and are central to agricultural economies.
Pigs are intelligent animals that can be trained to perform tasks.
Chickens are the most common domesticated animal and provide eggs.
Sheep produce wool and are raised in flocks on farms worldwide.
Goats are hardy animals that can survive in difficult mountain terrain.
The king and queen ruled the kingdom with wisdom and justice.
A man and a woman walked together through the quiet evening park.
The prince became king after the old ruler passed the crown.
She was a woman of great intelligence and remarkable courage.
The king ordered his soldiers to protect the kingdom from invaders.
A young man studied hard and became a doctor who saved lives.
The queen addressed her subjects with grace and great authority.
He was a man who dedicated his life to helping the poor.
The woman scientist discovered a cure that changed medicine forever.
The old king and wise woman advised the young prince together.
"""

# Разбиваем корпус на список предложений
sentences = [s.strip() for s in RAW_CORPUS.strip().split('\n') if s.strip()]
print(f'Всего предложений: {len(sentences)}')
print(f'Пример: "{sentences[0]}"')

## Токенизация и словарь

Строим словарь с порогом `MIN_FREQ` и, при желании, отбрасываем
слишком частые слова через subsampling.

In [ ]:
def tokenize(text: str) -> List[str]:
    """Простая токенизация: lowercase + удаление небуквенных символов."""
    text = text.lower()
    tokens = re.findall(r'[a-z]+', text)
    return tokens


class Vocabulary:
    """
    Словарь с поддержкой:
    - фильтрации по min_freq
    - unigram distribution для negative sampling
    - subsampling для частых слов
    """

    def __init__(self, min_freq: int = 2, ns_alpha: float = 0.75,
                 subsample_t: float = 1e-4):
        self.min_freq   = min_freq
        self.ns_alpha   = ns_alpha
        self.subsample_t = subsample_t

        # word <-> index
        self.word2idx: Dict[str, int] = {}
        self.idx2word: Dict[int, str] = {}
        self.word_freq: Dict[str, int] = {}

        # вероятности для negative sampling (сглаженное унирграм распределение)
        self.ns_probs: Optional[np.ndarray] = None
        # вероятности выбросить слово при subsampling (для частых слов)
        self.discard_probs: Optional[np.ndarray] = None

    def build(self, tokenized_sentences: List[List[str]]) -> None:
        """Строим словарь из токенизированных предложений."""
        # Подсчёт частот
        counter = collections.Counter()
        for tokens in tokenized_sentences:
            counter.update(tokens)

        # Фильтрация по min_freq
        filtered = [(w, c) for w, c in counter.items() if c >= self.min_freq]
        filtered.sort(key=lambda x: -x[1])  # по убыванию частоты

        for idx, (word, count) in enumerate(filtered):
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
            self.word_freq[word] = count

        print(f'Уникальных токенов в корпусе: {len(counter)}')
        print(f'Размер словаря (min_freq={self.min_freq}): {len(self.word2idx)}')

        self._build_ns_probs()
        self._build_discard_probs()

    def _build_ns_probs(self) -> None:
        """Сглаженное унирграм распределение P(w) ∝ freq(w)^alpha."""
        counts = np.array([self.word_freq[self.idx2word[i]]
                           for i in range(len(self))], dtype=np.float64)
        powered = counts ** self.ns_alpha
        self.ns_probs = powered / powered.sum()

    def _build_discard_probs(self) -> None:
        """P(discard|w) = 1 - sqrt(t / f(w)), где f(w) — нормированная частота."""
        total = sum(self.word_freq.values())
        self.discard_probs = np.zeros(len(self), dtype=np.float64)
        for idx in range(len(self)):
            freq = self.word_freq[self.idx2word[idx]] / total
            prob = 1.0 - math.sqrt(self.subsample_t / (freq + 1e-12))
            self.discard_probs[idx] = max(prob, 0.0)

    def sample_negatives(self, n: int) -> List[int]:
        """Сэмплируем n негативных индексов по unigram распределению."""
        return np.random.choice(len(self), size=n, p=self.ns_probs).tolist()

    def should_discard(self, idx: int) -> bool:
        """True если слово нужно выбросить при subsampling."""
        return random.random() < self.discard_probs[idx]

    def __len__(self) -> int:
        return len(self.word2idx)


# Токенизируем корпус
tokenized_sentences = [tokenize(s) for s in sentences]

# Строим словарь
vocab = Vocabulary(min_freq=MIN_FREQ, ns_alpha=NS_ALPHA, subsample_t=SUBSAMPLE_T)
vocab.build(tokenized_sentences)

VOCAB_SIZE = len(vocab)
print(f'\nТоп-10 слов: {[(vocab.idx2word[i], vocab.word_freq[vocab.idx2word[i]]) for i in range(10)]}')

## Выборка для Skip-gram

In [ ]:
class SkipGramDataset(Dataset):
    """
    Выборка для Skip-gram с Negative Sampling.

    Каждый элемент: (center_idx, pos_idx, neg_indices)
    - center_idx: индекс центрального слова
    - pos_idx:    индекс одного позитивного контекстного слова
    - neg_indices: список из NEG_SAMPLES случайных слов
    """

    def __init__(self, tokenized_sentences: List[List[str]],
                 vocab: Vocabulary,
                 window_size: int = WINDOW_SIZE,
                 neg_samples: int = NEG_SAMPLES,
                 use_subsampling: bool = USE_SUBSAMPLING):
        self.vocab = vocab
        self.window_size = window_size
        self.neg_samples = neg_samples
        self.pairs: List[Tuple[int, int]] = []
        self._build_pairs(tokenized_sentences, use_subsampling)
        print(f'[SkipGram] Сгенерировано пар (центр, контекст): {len(self.pairs)}')

    def _build_pairs(self, sentences: List[List[str]], use_subsampling: bool) -> None:
        for tokens in sentences:
            indices = [self.vocab.word2idx[t] for t in tokens if t in self.vocab.word2idx]
            if use_subsampling:
                indices = [idx for idx in indices if not self.vocab.should_discard(idx)]

            for i, center in enumerate(indices):
                win = random.randint(1, self.window_size)
                left = max(0, i - win)
                right = min(len(indices), i + win + 1)

                for j in range(left, right):
                    if j == i:
                        continue
                    self.pairs.append((center, indices[j]))

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        center, pos = self.pairs[idx]
        negatives = self.vocab.sample_negatives(self.neg_samples)
        negatives = [n if n != pos else (n + 1) % len(self.vocab) for n in negatives]
        return (
            torch.tensor(center, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
            torch.tensor(negatives, dtype=torch.long),
        )


sg_dataset = SkipGramDataset(
    tokenized_sentences,
    vocab,
    use_subsampling=USE_SUBSAMPLING,
)
sg_loader = DataLoader(
    sg_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

## Выборка для CBOW

In [ ]:
class CBOWDataset(Dataset):
    """
    Выборка для CBOW с Negative Sampling.

    Каждый элемент: (context_indices, center_idx, neg_indices)
    - context_indices: тензор индексов контекстных слов (длина 2*window)
    - center_idx:      индекс центрального (целевого) слова
    - neg_indices:     список из NEG_SAMPLES случайных слов
    """

    def __init__(self, tokenized_sentences: List[List[str]],
                 vocab: Vocabulary,
                 window_size: int = WINDOW_SIZE,
                 neg_samples: int = NEG_SAMPLES,
                 use_subsampling: bool = USE_SUBSAMPLING):
        self.vocab = vocab
        self.window_size = window_size
        self.neg_samples = neg_samples
        self.max_ctx_len = 2 * window_size
        self.pad_idx = len(vocab)
        self.samples: List[Tuple[List[int], int]] = []
        self._build_samples(tokenized_sentences, use_subsampling)
        print(f'[CBOW] Сгенерировано примеров (контекст, центр): {len(self.samples)}')

    def _build_samples(self, sentences: List[List[str]], use_subsampling: bool) -> None:
        for tokens in sentences:
            indices = [self.vocab.word2idx[t] for t in tokens if t in self.vocab.word2idx]
            if use_subsampling:
                indices = [idx for idx in indices if not self.vocab.should_discard(idx)]

            for i, center in enumerate(indices):
                win = random.randint(1, self.window_size)
                left = max(0, i - win)
                right = min(len(indices), i + win + 1)
                context = [indices[j] for j in range(left, right) if j != i]
                if context:
                    self.samples.append((context, center))

    def _pad_context(self, context: List[int]) -> List[int]:
        if len(context) >= self.max_ctx_len:
            return context[:self.max_ctx_len]
        return context + [self.pad_idx] * (self.max_ctx_len - len(context))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        context, center = self.samples[idx]
        negatives = self.vocab.sample_negatives(self.neg_samples)
        negatives = [n if n != center else (n + 1) % len(self.vocab) for n in negatives]
        return (
            torch.tensor(self._pad_context(context), dtype=torch.long),
            torch.tensor(center, dtype=torch.long),
            torch.tensor(negatives, dtype=torch.long),
        )


cbow_dataset = CBOWDataset(
    tokenized_sentences,
    vocab,
    use_subsampling=USE_SUBSAMPLING,
)
cbow_loader = DataLoader(
    cbow_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

## Модель Skip-gram

In [ ]:
class SkipGramModel(nn.Module):
    """
    Skip-gram с Negative Sampling.

    Две таблицы эмбеддингов:
      in_embed  — для центральных слов
      out_embed — для контекстных/негативных слов
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

        init_bound = 0.5 / embedding_dim
        nn.init.uniform_(self.in_embed.weight, -init_bound, init_bound)
        nn.init.uniform_(self.out_embed.weight, -init_bound, init_bound)

    def forward(self, center: torch.Tensor,
                pos: torch.Tensor,
                neg: torch.Tensor) -> torch.Tensor:
        v_c = self.in_embed(center)
        v_pos = self.out_embed(pos)
        pos_score = torch.sum(v_c * v_pos, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        v_neg = self.out_embed(neg)
        neg_score = torch.bmm(v_neg, v_c.unsqueeze(2)).squeeze(2)
        neg_loss = torch.sum(F.logsigmoid(-neg_score), dim=1)
        return -(pos_loss + neg_loss).mean()

    def get_embeddings(self) -> np.ndarray:
        return self.in_embed.weight.detach().cpu().numpy()


sg_model = SkipGramModel(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
print(sg_model)
total_params = sum(p.numel() for p in sg_model.parameters())
print(f'Параметров: {total_params:,}')

## Модель CBOW

In [ ]:
class CBOWModel(nn.Module):
    """
    CBOW с Negative Sampling.

    Контекстные векторы усредняются только по реальным словам,
    а не по padding-слотам.
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.pad_idx = vocab_size
        self.in_embed = nn.Embedding(vocab_size + 1, embedding_dim, padding_idx=self.pad_idx)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

        init_bound = 0.5 / embedding_dim
        nn.init.uniform_(self.in_embed.weight, -init_bound, init_bound)
        nn.init.uniform_(self.out_embed.weight, -init_bound, init_bound)
        with torch.no_grad():
            self.in_embed.weight[self.pad_idx].zero_()

    def forward(self, context: torch.Tensor,
                center: torch.Tensor,
                neg: torch.Tensor) -> torch.Tensor:
        ctx_vecs = self.in_embed(context)
        mask = (context != self.pad_idx).unsqueeze(-1)
        ctx_sum = (ctx_vecs * mask).sum(dim=1)
        ctx_count = mask.sum(dim=1).clamp_min(1)
        ctx_mean = ctx_sum / ctx_count

        v_center = self.out_embed(center)
        pos_score = torch.sum(ctx_mean * v_center, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        v_neg = self.out_embed(neg)
        neg_score = torch.bmm(v_neg, ctx_mean.unsqueeze(2)).squeeze(2)
        neg_loss = torch.sum(F.logsigmoid(-neg_score), dim=1)
        return -(pos_loss + neg_loss).mean()

    def get_embeddings(self) -> np.ndarray:
        return self.in_embed.weight[:self.pad_idx].detach().cpu().numpy()


cbow_model = CBOWModel(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
print(cbow_model)
total_params = sum(p.numel() for p in cbow_model.parameters())
print(f'Параметров: {total_params:,}')

## Hierarchical Softmax

In [ ]:
import heapq


class HuffmanNode:
    """Узел дерева Хаффмана."""

    def __init__(self, word_idx: int, freq: int):
        self.word_idx = word_idx
        self.freq = freq
        self.left = None
        self.right = None
        self.path: List[Tuple[int, int]] = []

    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_tree(vocab: Vocabulary) -> Dict[int, List[Tuple[int, int]]]:
    heap = [HuffmanNode(idx, vocab.word_freq[vocab.idx2word[idx]]) for idx in range(len(vocab))]
    heapq.heapify(heap)

    inner_node_counter = [0]
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        parent = HuffmanNode(-1, left.freq + right.freq)
        parent.left = left
        parent.right = right
        parent.word_idx = -(inner_node_counter[0] + 1)
        inner_node_counter[0] += 1
        heapq.heappush(heap, parent)

    root = heap[0]
    n_inner = inner_node_counter[0]
    word_paths: Dict[int, List[Tuple[int, int]]] = {}

    def traverse(node: HuffmanNode, path: List[Tuple[int, int]]):
        if node is None:
            return
        if node.word_idx >= 0:
            word_paths[node.word_idx] = path[:]
            return
        current_inner = abs(node.word_idx) - 1
        traverse(node.left, path + [(current_inner, 0)])
        traverse(node.right, path + [(current_inner, 1)])

    traverse(root, [])
    print(f'Дерево Хаффмана: {len(vocab)} листьев, {n_inner} внутренних узлов')
    avg_path = np.mean([len(path) for path in word_paths.values()])
    print(f'Средняя длина пути: {avg_path:.2f} (log2({len(vocab)}) = {math.log2(len(vocab)):.2f})')
    return word_paths, n_inner


class SkipGramHierarchicalSoftmax(nn.Module):
    """
    Skip-gram с Hierarchical Softmax.
    """

    def __init__(self, vocab_size: int, embedding_dim: int,
                 word_paths: Dict[int, List[Tuple[int, int]]],
                 n_inner_nodes: int):
        super().__init__()
        self.word_paths = word_paths
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.inner_vecs = nn.Embedding(n_inner_nodes, embedding_dim)

        init_bound = 0.5 / embedding_dim
        nn.init.uniform_(self.in_embed.weight, -init_bound, init_bound)
        nn.init.uniform_(self.inner_vecs.weight, -init_bound, init_bound)

    def forward_single(self, center_vec: torch.Tensor, target_word_idx: int) -> torch.Tensor:
        path = self.word_paths.get(target_word_idx, [])
        if not path:
            return center_vec.new_tensor(0.0)

        log_prob = center_vec.new_tensor(0.0)
        for inner_idx, direction in path:
            node_vec = self.inner_vecs.weight[inner_idx]
            score = torch.dot(center_vec, node_vec)
            log_prob = log_prob + (F.logsigmoid(score) if direction == 0 else F.logsigmoid(-score))
        return log_prob

    def forward(self, center: torch.Tensor,
                pos: torch.Tensor,
                neg: torch.Tensor = None) -> torch.Tensor:
        losses = []
        for i in range(center.size(0)):
            c_vec = self.in_embed(center[i])
            losses.append(-self.forward_single(c_vec, int(pos[i].item())))
        return torch.stack(losses).mean()

    def get_embeddings(self) -> np.ndarray:
        return self.in_embed.weight.detach().cpu().numpy()


word_paths, n_inner = build_huffman_tree(vocab)

hs_model = SkipGramHierarchicalSoftmax(
    VOCAB_SIZE, EMBEDDING_DIM, word_paths, n_inner
).to(DEVICE)

print(f'\nПуть к слову "king": {word_paths.get(vocab.word2idx.get("king", 0), [])[:5]}...')
print(f'Длина пути к слову "king": {len(word_paths.get(vocab.word2idx.get("king", 0), []))}')

## Цикл обучения

Для небольшого учебного корпуса используем `Adam` и мягкое снижение шага.
На таком объёме данных это ведёт себя спокойнее, чем прямой `SGD`.

In [ ]:
def train_model(model: nn.Module,
                loader: DataLoader,
                epochs: int = EPOCHS,
                lr: float = LEARNING_RATE,
                model_name: str = 'Model') -> List[float]:
    """
    Тренировочный цикл для небольшого корпуса.

    Adam здесь выбран осознанно: он быстрее уводит loss с плато
    на маленьком учебном датасете.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.97)
    epoch_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in loader:
            inp1, inp2, neg = [tensor.to(DEVICE) for tensor in batch]
            optimizer.zero_grad()
            loss = model(inp1, inp2, neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / max(n_batches, 1)
        epoch_losses.append(avg_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(
            f'[{model_name}] Epoch {epoch + 1:>2}/{epochs} | '
            f'Loss: {avg_loss:.4f} | lr: {current_lr:.5f}'
        )
        scheduler.step()

    return epoch_losses


print('=' * 55)
print('Обучение Skip-gram')
print('=' * 55)
sg_losses = train_model(sg_model, sg_loader, epochs=EPOCHS, model_name='SkipGram')

In [ ]:
print('=' * 55)
print('Обучение CBOW')
print('=' * 55)
cbow_losses = train_model(cbow_model, cbow_loader, epochs=EPOCHS, model_name='CBOW')

In [ ]:
print('=' * 55)
print('Обучение Hierarchical Softmax (Skip-gram)')
print('=' * 55)
hs_losses = train_model(
    hs_model,
    sg_loader,
    epochs=HS_EPOCHS,
    lr=LEARNING_RATE,
    model_name='HS-SkipGram',
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, EPOCHS + 1), sg_losses, 'b-o', ms=4, label='Skip-gram (NS)')
ax.plot(range(1, EPOCHS + 1), cbow_losses, 'r-s', ms=4, label='CBOW (NS)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss: Skip-gram vs CBOW')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, len(hs_losses) + 1), hs_losses, 'g-^', ms=6, label='Skip-gram (HS)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss: Hierarchical Softmax')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'word2vec_losses.png', dpi=120, bbox_inches='tight')
plt.show()

## Ближайшие соседи

In [ ]:
def _safe_l2_normalize(matrix: np.ndarray, axis: int = 1) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=axis, keepdims=True)
    safe_norms = np.where(norms < 1e-12, 1.0, norms)
    return matrix / safe_norms


def get_nearest_neighbors(word: str, embeddings: np.ndarray,
                          vocab: Vocabulary, top_k: int = 8) -> List[Tuple[str, float]]:
    """
    Находит top_k ближайших соседей слова по косинусному сходству.
    """
    if word not in vocab.word2idx:
        return []

    idx = vocab.word2idx[word]
    query_vec = embeddings[idx]
    if np.linalg.norm(query_vec) < 1e-12:
        return []

    normed = _safe_l2_normalize(embeddings, axis=1)
    q_normed = query_vec / max(np.linalg.norm(query_vec), 1e-12)
    sims = normed @ q_normed

    top_indices = np.argsort(sims)[::-1]
    results = []
    for i in top_indices:
        if i == idx or not np.isfinite(sims[i]):
            continue
        results.append((vocab.idx2word[i], float(sims[i])))
        if len(results) == top_k:
            break
    return results


def print_nearest_neighbors(words: List[str], model: nn.Module,
                             vocab: Vocabulary, model_name: str, top_k: int = 6):
    embeddings = model.get_embeddings()
    print(f'\n{"═" * 55}')
    print(f'  Ближайшие соседи — {model_name}')
    print(f'{"═" * 55}')
    for word in words:
        neighbors = get_nearest_neighbors(word, embeddings, vocab, top_k)
        if not neighbors:
            print(f'  "{word}" не найдено в словаре или имеет нулевой вектор')
            continue
        nb_str = ', '.join([f'{w} ({s:.3f})' for w, s in neighbors])
        print(f'  {word:<12}: {nb_str}')


query_words = ['king', 'woman', 'ocean', 'lion',
               'learning', 'earth', 'forest', 'human', 'language']
query_words = [word for word in query_words if word in vocab.word2idx]
print(f'Запросные слова (в словаре): {query_words}')

print_nearest_neighbors(query_words, sg_model, vocab, 'Skip-gram (NS)')
print_nearest_neighbors(query_words, cbow_model, vocab, 'CBOW (NS)')

## Аналогии

In [ ]:
def analogy(word_a: str, word_b: str, word_c: str,
            embeddings: np.ndarray, vocab: Vocabulary,
            top_k: int = 5) -> List[Tuple[str, float]]:
    """
    Решаем аналогию: word_a - word_b + word_c = ?
    """
    for word in [word_a, word_b, word_c]:
        if word not in vocab.word2idx:
            return [f'Слово "{word}" не найдено в словаре']

    normed = _safe_l2_normalize(embeddings, axis=1)
    va = normed[vocab.word2idx[word_a]]
    vb = normed[vocab.word2idx[word_b]]
    vc = normed[vocab.word2idx[word_c]]

    target = va - vb + vc
    target_norm = np.linalg.norm(target)
    if target_norm < 1e-12:
        return ['Целевой вектор аналогии выродился в ноль']
    target = target / target_norm

    sims = normed @ target
    exclude = {vocab.word2idx[word] for word in [word_a, word_b, word_c]}
    top_indices = np.argsort(sims)[::-1]

    results = []
    for i in top_indices:
        if i in exclude or not np.isfinite(sims[i]):
            continue
        results.append((vocab.idx2word[i], float(sims[i])))
        if len(results) == top_k:
            break
    return results


def print_analogies(analogy_tests: List[Tuple], model: nn.Module,
                    vocab: Vocabulary, model_name: str):
    embeddings = model.get_embeddings()
    print(f'\n{"═" * 65}')
    print(f'  Аналогии — {model_name}')
    print(f'{"═" * 65}')
    for a, b, c, expected in analogy_tests:
        results = analogy(a, b, c, embeddings, vocab, top_k=5)
        if isinstance(results[0], str):
            print(f'  {a} - {b} + {c} = {results[0]}')
            continue
        top5 = [f'{w} ({s:.3f})' for w, s in results]
        found = any(w == expected for w, _ in results)
        marker = '  [+]' if found else '  [ ]'
        print(f'{marker} {a} - {b} + {c} = ? (ожидаем: {expected})')
        print(f'       Топ-5: {" | ".join(top5)}')


analogy_tests = [
    ('king', 'man', 'woman', 'queen'),
    ('king', 'queen', 'man', 'woman'),
    ('lion', 'africa', 'ocean', 'whale'),
    ('wolf', 'forest', 'ocean', 'whale'),
    ('dog', 'animal', 'human', 'person'),
    ('learning', 'network', 'forest', 'ecosystem'),
    ('earth', 'planet', 'ocean', 'water'),
]

print_analogies(analogy_tests, sg_model, vocab, 'Skip-gram (NS)')
print_analogies(analogy_tests, cbow_model, vocab, 'CBOW (NS)')

print('\n[+] = ожидаемое слово найдено в топ-5')
print('[Примечание] Малый корпус снижает качество аналогий — это ожидаемо.')

## Визуализация через PCA

In [ ]:
def plot_pca(model: nn.Module, vocab: Vocabulary, model_name: str,
             n_words: int = 60, highlight_groups: Dict[str, List[str]] = None,
             ax=None):
    """
    PCA-визуализация эмбеддингов.

    Args:
        n_words: сколько самых частых слов отобразить
        highlight_groups: {label: [words]} для выделения кластеров цветом
    """
    embeddings = model.get_embeddings()

    # Берём топ-N самых частых слов из словаря
    top_words   = [vocab.idx2word[i] for i in range(min(n_words, len(vocab)))]
    top_indices = [vocab.word2idx[w] for w in top_words]
    top_emb     = embeddings[top_indices]

    # PCA: 100 → 2
    pca    = PCA(n_components=2, random_state=SEED)
    coords = pca.fit_transform(top_emb)

    explained = pca.explained_variance_ratio_

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 9))

    ax.set_title(f'PCA — {model_name}\n'
                 f'Объяснённая дисперсия: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}',
                 fontsize=11)

    # Базовые точки
    ax.scatter(coords[:, 0], coords[:, 1], c='lightgray', s=20, alpha=0.5, zorder=1)

    # Выделяем тематические группы
    if highlight_groups:
        colors = cm.tab10(np.linspace(0, 1, len(highlight_groups)))
        for (label, group_words), color in zip(highlight_groups.items(), colors):
            for word in group_words:
                if word not in vocab.word2idx or word not in top_words:
                    continue
                local_idx = top_words.index(word)
                ax.scatter(coords[local_idx, 0], coords[local_idx, 1],
                           c=[color], s=80, zorder=3, label=label)
                ax.annotate(word, (coords[local_idx, 0], coords[local_idx, 1]),
                            fontsize=8, ha='center', va='bottom',
                            xytext=(0, 4), textcoords='offset points')

        # Убираем дублирующиеся легенды
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='best')
    else:
        # Показываем все подписи
        for i, word in enumerate(top_words):
            ax.annotate(word, (coords[i, 0], coords[i, 1]),
                        fontsize=7, alpha=0.7)

    ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
    ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
    ax.grid(True, alpha=0.2)


# Тематические группы для подсветки
highlight_groups = {
    'Animals':    ['lion', 'wolf', 'dog', 'bird', 'whale', 'tiger', 'bear', 'elephant', 'fish'],
    'Nature':     ['ocean', 'forest', 'mountain', 'river', 'earth', 'water', 'sun', 'tree'],
    'Technology': ['network', 'learning', 'model', 'data', 'computer', 'language', 'algorithm'],
    'Royalty':    ['king', 'queen', 'man', 'woman', 'prince'],
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plot_pca(sg_model,   vocab, 'Skip-gram (NS)', n_words=80,
         highlight_groups=highlight_groups, ax=axes[0])
plot_pca(cbow_model, vocab, 'CBOW (NS)',      n_words=80,
         highlight_groups=highlight_groups, ax=axes[1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'word2vec_pca.png', dpi=130, bbox_inches='tight')
plt.show()

## Визуализация через t-SNE

In [ ]:
def plot_tsne(model: nn.Module, vocab: Vocabulary, model_name: str,
              n_words: int = 80, highlight_groups: Dict[str, List[str]] = None,
              perplexity: int = 20, ax=None):
    """
    t-SNE визуализация эмбеддингов.

    Args:
        perplexity: параметр t-SNE, обычно 5-50
                    (должен быть < числа точек)
    """
    embeddings = model.get_embeddings()

    top_words   = [vocab.idx2word[i] for i in range(min(n_words, len(vocab)))]
    top_indices = [vocab.word2idx[w] for w in top_words]
    top_emb     = embeddings[top_indices]

    # Шаг 1: PCA до 50 (или dim если меньше) — ускоряет t-SNE
    pre_dim = min(50, top_emb.shape[0] - 1, top_emb.shape[1])
    if pre_dim > 2:
        pca_pre = PCA(n_components=pre_dim, random_state=SEED)
        top_emb = pca_pre.fit_transform(top_emb)

    # Шаг 2: t-SNE до 2D
    perplexity = min(perplexity, len(top_words) - 1)  # perplexity < n_samples
    tsne   = TSNE(n_components=2, perplexity=perplexity,
                  max_iter=1000, random_state=SEED, verbose=0)
    coords = tsne.fit_transform(top_emb)

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 9))

    ax.set_title(f't-SNE — {model_name}\n'
                 f'(perplexity={perplexity}, n_words={len(top_words)})', fontsize=11)

    ax.scatter(coords[:, 0], coords[:, 1], c='lightgray', s=20, alpha=0.4, zorder=1)

    if highlight_groups:
        colors = cm.tab10(np.linspace(0, 1, len(highlight_groups)))
        for (label, group_words), color in zip(highlight_groups.items(), colors):
            for word in group_words:
                if word not in vocab.word2idx or word not in top_words:
                    continue
                local_idx = top_words.index(word)
                ax.scatter(coords[local_idx, 0], coords[local_idx, 1],
                           c=[color], s=100, zorder=3, label=label)
                ax.annotate(word, (coords[local_idx, 0], coords[local_idx, 1]),
                            fontsize=8, ha='center', va='bottom',
                            xytext=(0, 4), textcoords='offset points')

        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='best')
    else:
        for i, word in enumerate(top_words):
            ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.7)

    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.grid(True, alpha=0.2)


print('Запуск t-SNE (это займёт ~30 секунд)...')
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plot_tsne(sg_model,   vocab, 'Skip-gram (NS)', n_words=80,
          highlight_groups=highlight_groups, perplexity=20, ax=axes[0])
plot_tsne(cbow_model, vocab, 'CBOW (NS)',      n_words=80,
          highlight_groups=highlight_groups, perplexity=20, ax=axes[1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'word2vec_tsne.png', dpi=130, bbox_inches='tight')
plt.show()

## Сопоставление Skip-gram и CBOW

In [ ]:
def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    """Косинусное сходство двух векторов."""
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10))


def evaluate_word_pairs(word_pairs: List[Tuple[str, str, str]],
                        model: nn.Module, vocab: Vocabulary) -> Dict:
    """
    Оцениваем качество эмбеддингов через набор пар слов.

    word_pairs: [(word1, word2, relation)] где relation — тип смысловой связи

    Возвращаем средние косинусные сходства по типам связей.
    """
    embeddings = model.get_embeddings()
    results    = collections.defaultdict(list)

    for w1, w2, relation in word_pairs:
        if w1 not in vocab.word2idx or w2 not in vocab.word2idx:
            continue
        v1  = embeddings[vocab.word2idx[w1]]
        v2  = embeddings[vocab.word2idx[w2]]
        sim = cosine_similarity(v1, v2)
        results[relation].append((w1, w2, sim))

    return dict(results)


# Семантически связанные пары
word_pairs = [
    # Животные
    ('lion',   'tiger',    'похожие животные'),
    ('dog',    'wolf',     'похожие животные'),
    ('whale',  'dolphin',  'похожие животные'),
    # Природа
    ('ocean',  'sea',      'близкая природа'),
    ('forest', 'tree',     'связанная природа'),
    ('mountain', 'river',  'связанная природа'),
    # Технологии
    ('network', 'model',   'связанная техника'),
    ('learning', 'data',   'связанная техника'),
    # Несвязанные
    ('lion',   'network',  'несвязанные'),
    ('ocean',  'learning', 'несвязанные'),
    ('tree',   'model',    'несвязанные'),
]

print('\n' + '=' * 65)
print('  Сравнение средних косинусных сходств по категориям')
print('=' * 65)

for model, name in [(sg_model, 'Skip-gram'), (cbow_model, 'CBOW')]:
    results = evaluate_word_pairs(word_pairs, model, vocab)
    print(f'\n  [{name}]')
    for relation, pairs in sorted(results.items()):
        avg_sim = np.mean([s for _, _, s in pairs])
        print(f'    {relation:<20}: ср. cosine = {avg_sim:.4f}  '
              f'({len(pairs)} пар)')
        for w1, w2, s in pairs:
            print(f'      {w1} ~ {w2}: {s:.4f}')

In [ ]:
# Итоговый сравнительный дашборд
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Word2Vec: Skip-gram и CBOW рядом', fontsize=14, fontweight='bold')

# 1. Кривые потерь
ax = axes[0, 0]
ax.plot(range(1, EPOCHS + 1), sg_losses,   'b-o', ms=4, label='Skip-gram')
ax.plot(range(1, EPOCHS + 1), cbow_losses, 'r-s', ms=4, label='CBOW')
ax.set_xlabel('Эпоха')
ax.set_ylabel('Потери NS')
ax.set_title('Потери при обучении')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Косинусные сходства по грубым категориям
ax = axes[0, 1]
categories = ['похожие животные', 'связанная природа', 'связанная техника', 'несвязанные']
sg_avgs = []
cbow_avgs = []
for cat in categories:
    sg_res = evaluate_word_pairs(word_pairs, sg_model, vocab)
    cbow_res = evaluate_word_pairs(word_pairs, cbow_model, vocab)
    sg_avgs.append(np.mean([s for _, _, s in sg_res.get(cat, [('', '', 0)])]) if cat in sg_res else 0)
    cbow_avgs.append(np.mean([s for _, _, s in cbow_res.get(cat, [('', '', 0)])]) if cat in cbow_res else 0)

x = np.arange(len(categories))
width = 0.35
ax.bar(x - width / 2, sg_avgs, width, label='Skip-gram', color='steelblue', alpha=0.8)
ax.bar(x + width / 2, cbow_avgs, width, label='CBOW', color='tomato', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['животные', 'природа', 'техника', 'фон'], fontsize=8)
ax.set_ylabel('Среднее косинусное сходство')
ax.set_title('Смысловая близость по группам')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linewidth=0.8)

# 3. Распределение норм векторов
ax = axes[1, 0]
sg_norms = np.linalg.norm(sg_model.get_embeddings(), axis=1)
cbow_norms = np.linalg.norm(cbow_model.get_embeddings(), axis=1)
ax.hist(sg_norms, bins=30, alpha=0.6, label='Skip-gram', color='steelblue')
ax.hist(cbow_norms, bins=30, alpha=0.6, label='CBOW', color='tomato')
ax.set_xlabel('L2-норма вектора')
ax.set_ylabel('Частота')
ax.set_title('Распределение норм эмбеддингов')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Небольшая сводка
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Характеристика', 'Skip-gram', 'CBOW'],
    ['Направление', 'центр → контекст', 'контекст → центр'],
    ['Примеров для обучения', f'{len(sg_dataset):,}', f'{len(cbow_dataset):,}'],
    ['Финальные потери', f'{sg_losses[-1]:.4f}', f'{cbow_losses[-1]:.4f}'],
    ['Скорость', 'Медленнее', 'Быстрее'],
    ['Редкие слова', 'Обычно лучше', 'Обычно слабее'],
    ['Частые слова', 'Сравнимо', 'Сравнимо'],
    ['Средняя норма', f'{sg_norms.mean():.3f}', f'{cbow_norms.mean():.3f}'],
]
table = ax.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for j in range(3):
    table[(0, j)].set_facecolor('#4472C4')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title('Короткая сводка', pad=20, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'word2vec_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

print('\nСводный рисунок сохранён как word2vec_comparison.png')

In [ ]:
print('=' * 65)
print('ИТОГ ПО WORD2VEC')
print('=' * 65)
print(f'''
Корпус:
  Предложений:    {len(sentences)}
  Размер словаря: {VOCAB_SIZE} слов (min_freq={MIN_FREQ})

Параметры:
  embedding_dim:  {EMBEDDING_DIM}
  window_size:    {WINDOW_SIZE}
  negative_samples: {NEG_SAMPLES}
  epochs:         {EPOCHS}
  batch_size:     {BATCH_SIZE}

Обучение:
  Skip-gram пар:  {len(sg_dataset):,}
  CBOW примеров:  {len(cbow_dataset):,}
  Skip-gram loss: {sg_losses[-1]:.4f}
  CBOW loss:      {cbow_losses[-1]:.4f}
''')

print('Что видно по результату:')
print('  - Skip-gram генерирует больше учебных пар и обычно лучше держится на редких словах.')
print('  - CBOW быстрее по шагу и в небольших задачах часто оказывается удобнее для чернового прогона.')
print('  - На маленьком корпусе аналогии ограничены объёмом текста, так что чудес здесь ждать не стоит.')
print('  - Для реальной задачи такой код удобен как учебная реализация, а не как замена готовым библиотекам.')